# A Abordagem de Feature Store

Para organizar as variáveis utilizadas no modelo, foi adotada uma abordagem baseada em **Feature Store**. O objetivo é separar a **construção das features** da etapa de modelagem, permitindo que as mesmas variáveis sejam utilizadas de forma consistente tanto no treinamento quanto na previsão.

Construção das features:

A criação das variáveis é realizada utilizando **queries SQL**, armazenadas em arquivos *.sql* separados. Essa organização permite manter a lógica de cada grupo de features isolada e facilita sua manutenção e reutilização.

A estrutura segue, por exemplo:

```text
feature_store/
├── fs_temporal.sql
├── fs_loja.sql
├── fs_vendas.sql
└── fs_clientes.sql
```

As queries são parametrizadas pelo período de referência. Dessa forma, a mesma consulta pode ser executada para diferentes meses, gerando as features correspondentes a cada DATA_REF.

Ingestão no Feature Store:

A execução das queries é centralizada em um **notebook de ingestão**. Esse notebook lê cada arquivo *.sql*, executa a consulta para os períodos definidos e grava os resultados nas respectivas tabelas do Feature Store.

O fluxo pode ser representado da seguinte forma:

```text
Dados brutos
     ↓
Queries SQL
     ↓
Construção das features
     ↓
Notebook de ingestão
     ↓
Feature Store
     ↓
Training Set / Predição
```

Na primeira execução, caso a tabela ainda não exista, ela é criada definindo:

* *STORE*
* *DATA_REF*

como chaves da feature;

* *DATA_REF* como coluna de particionamento.

Nas execuções seguintes, os novos períodos são adicionados utilizando **merge**, permitindo atualizar o Feature Store sem precisar recriar toda a tabela.

> **Importante:** como se trata de um problema temporal, as features históricas devem ser construídas utilizando apenas informações disponíveis até a respectiva *DATA_REF*. Isso evita que informações futuras sejam utilizadas durante o treinamento e reduz o risco de *data leakage*.



As features foram divididas em diferentes grupos de acordo com sua origem e finalidade:

* **Temporal**: Features relacionadas ao funcionamento da loja, promoções, feriados e eventos temporais observados até a data de referência;

* **Loja**: Features relacionadas às características estruturais da loja, à concorrência e à participação em programas promocionais;

* **Vendas**: Features relacionadas ao histórico, comportamento, sazonalidade e tendência das vendas da loja;

* **Clientes**: Features relacionadas ao comportamento e ao volume de clientes da loja;



## Feature Store Temporal


**Chave:** ID_LOJA + REF_DATE

Features relacionadas ao funcionamento da loja, promoções, feriados e eventos temporais observados até a data de referência.


**Features**

* **Dias com loja aberta:** quantidade de dias em que a loja permaneceu aberta nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Dias com loja fechada:** quantidade de dias em que a loja permaneceu fechada nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Taxa de funcionamento:** proporção de dias em que a loja permaneceu aberta nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Dias com promoção:** quantidade de dias em que a loja esteve em promoção nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Dias sem promoção:** quantidade de dias em que a loja não esteve em promoção nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Taxa de promoção:** proporção de dias em que a loja esteve em promoção nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Dias com feriado estadual:** quantidade de dias com feriado estadual nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Taxa de feriados estaduais:** proporção de dias com feriado estadual nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Dias com feriado escolar:** quantidade de dias com feriado escolar nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Taxa de feriados escolares:** proporção de dias com feriado escolar nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Tempo desde o início da competição:** quantidade de dias decorridos entre a data de abertura da concorrência e a data de referência.

* **Tempo desde o início da Promo2:** quantidade de dias decorridos entre a data de início da Promo2 e a data de referência.


## Feature Store Loja



**Chave:** ID_LOJA + REF_DATE

Features relacionadas às características estruturais da loja, à concorrência e à participação em programas promocionais.



**Features**

* **Competição ativa:** indicador que informa se havia uma concorrência ativa para a loja na data de referência.

* **Promo2 ativa:** indicador que informa se a loja estava participando de uma Promo2 ativa na data de referência.

* **Tipo da loja:** classificação da loja de acordo com o seu tipo.

* **Tipo de sortimento:** classificação do sortimento de produtos oferecido pela loja.

* **Distância até a concorrência:** distância, em metros, entre a loja e a concorrência mais próxima.


## Feature Store Vendas

**Chave:** ID_LOJA + REF_DATE

Features relacionadas ao histórico, comportamento, sazonalidade e tendência das vendas da loja, considerando apenas as informações disponíveis até a data de referência.

**Features**

* **Soma de vendas:** valor total das vendas nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Média de vendas:** média diária das vendas nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Mínimo de vendas:** menor valor diário de vendas nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Máximo de vendas:** maior valor diário de vendas nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Desvio padrão de vendas:** medida da variabilidade diária das vendas nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Mediana de vendas:** valor central das vendas diárias nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Soma de vendas durante promoção:** valor total das vendas nos dias em que a loja esteve em promoção nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Média de vendas durante promoção:** média diária das vendas nos dias em que a loja esteve em promoção nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Soma de vendas sem promoção:** valor total das vendas nos dias em que a loja não esteve em promoção nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Média de vendas sem promoção:** média diária das vendas nos dias em que a loja não esteve em promoção nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Lift de vendas durante promoção:** diferença percentual entre a média de vendas durante períodos com promoção e a média de vendas durante períodos sem promoção nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Média de vendas do mesmo período do ano anterior:** média das vendas nos 42 dias equivalentes do ano anterior.

* **Crescimento em relação ao ano anterior:** variação percentual da média de vendas dos últimos 42 dias em relação à média de vendas do período equivalente do ano anterior.

* **Soma de vendas por dia da semana:** soma das vendas para cada dia da semana considerando as últimas 4, 8 e 12 ocorrências daquele mesmo dia da semana.

* **Média de vendas por dia da semana:** média das vendas para cada dia da semana considerando as últimas 4, 8 e 12 ocorrências daquele mesmo dia da semana.

* **Desvio padrão de vendas por dia da semana:** variabilidade das vendas para cada dia da semana considerando as últimas 4, 8 e 12 ocorrências daquele mesmo dia da semana.

* **Vendas por cliente:** relação entre o total de vendas e o total de clientes nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Histórico diário de vendas:** valores individuais das vendas observadas nos últimos 84 dias anteriores à data de referência.

* **Crescimento de vendas 7D vs. 28D:** crescimento percentual da média de vendas dos últimos 7 dias em relação à média dos últimos 28 dias.

* **Crescimento de vendas 14D vs. 28D:** crescimento percentual da média de vendas dos últimos 14 dias em relação à média dos últimos 28 dias.

* **Crescimento de vendas 28D vs. 56D:** crescimento percentual da média de vendas dos últimos 28 dias em relação à média dos últimos 56 dias.

* **Crescimento de vendas 42D vs. 84D:** crescimento percentual da média de vendas dos últimos 42 dias em relação à média dos últimos 84 dias.


## Feature Store Clientes



**Chave:** ID_LOJA + REF_DATE

Features relacionadas ao comportamento e ao volume de clientes da loja, considerando o histórico observado até a data de referência.


**Features**

* **Soma de clientes:** quantidade total de clientes nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Média de clientes:** média diária de clientes nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Mínimo de clientes:** menor número diário de clientes nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Máximo de clientes:** maior número diário de clientes nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Desvio padrão de clientes:** variabilidade diária do número de clientes nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Mediana de clientes:** mediana do número diário de clientes nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Soma de clientes durante promoção:** quantidade total de clientes nos dias em que a loja esteve em promoção nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Média de clientes durante promoção:** média diária de clientes nos dias em que a loja esteve em promoção nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Soma de clientes sem promoção:** quantidade total de clientes nos dias em que a loja não esteve em promoção nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Média de clientes sem promoção:** média diária de clientes nos dias em que a loja não esteve em promoção nos últimos 7, 14, 28, 42, 56 e 84 dias.

* **Crescimento de clientes 7D vs. 28D:** crescimento percentual da média de clientes dos últimos 7 dias em relação aos últimos 28 dias.

* **Crescimento de clientes 14D vs. 28D:** crescimento percentual da média de clientes dos últimos 14 dias em relação aos últimos 28 dias.

* **Crescimento de clientes 28D vs. 56D:** crescimento percentual da média de clientes dos últimos 28 dias em relação aos últimos 56 dias.

* **Crescimento de clientes 42D vs. 84D:** crescimento percentual da média de clientes dos últimos 42 dias em relação aos últimos 84 dias.


## Observação 

> Essas são as features inicialmente propostas. A definição final poderá ser alterada durante a exploração e modelagem, conforme a disponibilidade das informações, relevância preditiva e necessidade de evitar *data leakage*.